# 부록 04. create_agent로 간단한 에이전트 만들기

LangChain 1.0의 `create_agent` 함수를 사용하여 프로덕션 수준의 에이전트를 쉽게 만들 수 있습니다.

## 학습 목표
- `create_agent` 함수의 핵심 파라미터 이해
- 다양한 설정으로 에이전트 커스터마이징
- ReAct 패턴의 작동 원리 이해

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. create_agent 기본 사용법

`create_agent`는 LangGraph 기반의 에이전트를 생성합니다.

### 핵심 파라미터
- `model`: 사용할 LLM (문자열 또는 모델 인스턴스)
- `tools`: 에이전트가 사용할 도구 리스트
- `system_prompt`: 에이전트의 행동을 안내하는 시스템 프롬프트

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# 간단한 도구 정의
@tool
def get_weather(city: str) -> str:
    """특정 도시의 날씨 정보를 가져옵니다."""
    weather_db = {
        "서울": "맑음, 22°C",
        "도쿄": "흐림, 18°C",
        "뉴욕": "비, 15°C"
    }
    return weather_db.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# 가장 기본적인 에이전트 생성
agent = create_agent(
    model="gpt-4.1-mini",
    tools=[get_weather],
    system_prompt="당신은 날씨 정보를 제공하는 도우미입니다."
)

print("에이전트가 생성되었습니다.")

In [ ]:
# 에이전트 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "서울 날씨 어때?"}]
})

print("=== 에이전트 응답 ===")
print(result["messages"][-1].content)

## 3. ReAct 패턴 이해하기

에이전트는 **ReAct** (Reasoning + Acting) 패턴을 따릅니다:
1. **생각(Reasoning)**: 무엇을 해야 할지 결정
2. **행동(Acting)**: 도구 호출
3. **관찰(Observation)**: 결과 확인
4. 필요시 1-3 반복

In [ ]:
# ReAct 패턴을 볼 수 있도록 전체 메시지 출력
result = agent.invoke({
    "messages": [{"role": "user", "content": "도쿄 날씨 알려줘"}]
})

print("=== 전체 메시지 히스토리 ===")
for i, msg in enumerate(result["messages"]):
    msg_type = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"[{i}] {msg_type}: 도구 호출 - {msg.tool_calls[0]['name']}")
    elif hasattr(msg, 'name'):
        print(f"[{i}] {msg_type} (도구: {msg.name}): {msg.content[:100]}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"[{i}] {msg_type}: {content[:100]}..." if len(content) > 100 else f"[{i}] {msg_type}: {content}")

## 4. 여러 도구를 가진 에이전트

In [ ]:
from datetime import datetime

@tool
def get_current_time() -> str:
    """현재 시간을 반환합니다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. 예: '10 * 5 + 3'"""
    try:
        return str(eval(expression))
    except:
        return "계산할 수 없는 표현식입니다."

@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾았습니다. (시뮬레이션)"

# 다중 도구 에이전트
multi_tool_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[get_current_time, calculate, search_web, get_weather],
    system_prompt="""당신은 다재다능한 어시스턴트입니다.
- 시간 관련 질문: get_current_time 사용
- 계산 질문: calculate 사용
- 검색 필요: search_web 사용
- 날씨 질문: get_weather 사용
항상 한국어로 친절하게 답변하세요."""
)

print("다중 도구 에이전트가 생성되었습니다.")

In [ ]:
# 다양한 질문 테스트
questions = [
    "지금 몇 시야?",
    "256 곱하기 128은?",
    "서울 날씨랑 현재 시간 알려줘"
]

for q in questions:
    result = multi_tool_agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"Q: {q}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 60)

## 5. 시스템 프롬프트 설계

좋은 시스템 프롬프트는 에이전트의 행동을 명확하게 안내합니다.

In [ ]:
# 역할 기반 시스템 프롬프트
CUSTOMER_SERVICE_PROMPT = """당신은 온라인 쇼핑몰의 고객 서비스 담당자입니다.

<역할>
- 고객 문의에 친절하고 전문적으로 응답
- 제품, 배송, 반품 관련 질문 처리
- 필요시 도구를 사용하여 정보 조회
</역할>

<톤앤매너>
- 항상 존댓말 사용
- 긍정적이고 해결 지향적
- 명확하고 간결한 답변
</톤앤매너>

<제한사항>
- 개인정보 요청 금지
- 회사 정책 외의 약속 금지
</제한사항>"""

@tool
def check_order_status(order_id: str) -> str:
    """주문 상태를 확인합니다."""
    orders = {
        "A12345": "배송 중 (내일 도착 예정)",
        "B67890": "상품 준비 중",
        "C11111": "배송 완료"
    }
    return orders.get(order_id, f"주문번호 {order_id}를 찾을 수 없습니다.")

@tool
def get_product_info(product_name: str) -> str:
    """제품 정보를 조회합니다."""
    products = {
        "무선 이어폰": "가격: 89,000원, 재고: 있음, 배송: 1-2일",
        "노트북 거치대": "가격: 35,000원, 재고: 품절, 입고예정: 다음주"
    }
    return products.get(product_name, f"{product_name} 정보를 찾을 수 없습니다.")

cs_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[check_order_status, get_product_info],
    system_prompt=CUSTOMER_SERVICE_PROMPT
)

print("고객 서비스 에이전트가 생성되었습니다.")

In [ ]:
# 고객 서비스 테스트
cs_queries = [
    "주문번호 A12345 배송 상태 확인해주세요",
    "무선 이어폰 재고 있나요?",
    "반품하고 싶은데 어떻게 하나요?"
]

for query in cs_queries:
    result = cs_agent.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"고객: {query}")
    print(f"상담원: {result['messages'][-1].content}")
    print("=" * 60)

## 6. 모델 인스턴스 직접 전달

In [ ]:
from langchain.chat_models import init_chat_model

# 모델을 먼저 설정
model = init_chat_model(
    "gpt-4.1-mini",
    temperature=0,      # 결정적 응답
    max_tokens=500,     # 응답 길이 제한
)

# 모델 인스턴스를 에이전트에 전달
precise_agent = create_agent(
    model=model,
    tools=[calculate],
    system_prompt="정확한 계산을 제공하는 수학 도우미입니다. 결과만 간결하게 답변하세요."
)

result = precise_agent.invoke({
    "messages": [{"role": "user", "content": "15 * 23 + 100 - 45 계산해줘"}]
})
print(result["messages"][-1].content)

## 7. 도구 없는 에이전트

도구 없이 순수 대화형 에이전트도 만들 수 있습니다.

In [ ]:
# 도구 없는 대화형 에이전트
chat_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[],  # 빈 리스트
    system_prompt="""당신은 친절한 한국어 대화 상대입니다.
사용자와 자연스럽게 대화하며, 다양한 주제에 대해 이야기할 수 있습니다.
너무 길지 않게, 2-3문장으로 답변하세요."""
)

result = chat_agent.invoke({
    "messages": [{"role": "user", "content": "오늘 기분이 좀 우울해"}]
})
print(result["messages"][-1].content)

## 8. 요약

### 이 노트북에서 배운 것

1. **create_agent 기본 사용법**
   ```python
   agent = create_agent(
       model="gpt-4.1-mini",
       tools=[tool1, tool2],
       system_prompt="..."
   )
   ```

2. **ReAct 패턴**
   - 생각 → 행동 → 관찰 → 반복

3. **시스템 프롬프트 설계**
   - 역할, 톤앤매너, 제한사항 명시
   - XML 태그로 구조화 가능

4. **모델 설정**
   - 문자열 또는 모델 인스턴스 전달 가능
   - temperature, max_tokens 등 조절

### 다음 단계
- 부록_05: StateGraph API로 커스텀 그래프 구축하기